# Simulador Investopedia: P&L realizado

**Proyecto Final · Productos Derivados y Administración de Riesgos Corporativos**  
EGADE Business School · Equipo 6 · Grupo 40  
Eduardo Ayala Saldaña, Mariann Antonelli Morales Tobias, Ulises Salinas López  
Dr. Raymundo Díaz Robles · Noviembre de 2026

Origen: `06_codigo_python/pnl_edu.py`

```
Reconstruccion del P&L realizado de Eduardo Ayala (A01192714) en el
simulador Investopedia, del 22-sep al 20-oct de 2026.

Los simbolos siguen la convencion OCC abreviada:
   TICKER + AA + DD + letra_de_mes + strike
La letra codifica mes Y tipo: A-L son calls de enero a diciembre,
M-X son puts de enero a diciembre. Asi J = call de octubre y
V = put de octubre.
```

In [1]:
# Carpeta de datos (09_datos del repositorio). En Colab, si no esta presente,
# se pide subir el archivo necesario.
import os
DATA = next((p for p in ["../../09_datos", "../09_datos", "09_datos", "/content/09_datos"] if os.path.isfile(os.path.join(p, "historial_investopedia.html"))), None)
if DATA is None:
    from google.colab import files
    os.makedirs("/content/09_datos", exist_ok=True)
    print("Sube historial_investopedia.html")
    up = files.upload()
    for k, v in up.items():
        open(os.path.join("/content/09_datos", k), "wb").write(v)
    DATA = "/content/09_datos"
print("Datos en:", DATA)

Datos en: ../../09_datos


In [2]:
import sys, re, pandas as pd
from collections import defaultdict

MES = {}
for i,l in enumerate("ABCDEFGHIJKL"): MES[l] = (i+1, "call")
for i,l in enumerate("MNOPQRSTUVWX"): MES[l] = (i+1, "put")
NOM = ["","ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]

In [3]:
def decod(s):
    m = re.match(r"^([A-Z]+)(\d{2})(\d{2})([A-Z])([\d.]+)$", s)
    if not m: return None
    tk, aa, dd, L, k = m.groups()
    mes, tipo = MES[L]
    return dict(tk=tk, venc=f"{int(dd)}-{NOM[mes]}-20{aa}", tipo=tipo, k=float(k))

In [4]:
df = pd.read_html(f'{DATA}/historial_investopedia.html')[0]
df["Price"] = df["Price"].str.replace(r"[$,]", "", regex=True).astype(float)
df["Date"]  = pd.to_datetime(df["Date"])
df = df.sort_values("Date", kind="stable")

In [5]:
print("="*96)
print("A. OPERACIONES DECODIFICADAS")

A. OPERACIONES DECODIFICADAS


In [6]:
print("="*96)
print(f"   {'fecha':>11} {'subyacente':>6} {'tipo':>5} {'strike':>8} {'vence':>12} "
      f"{'accion':>8} {'cant':>5} {'precio':>8} {'efectivo':>11}")
pos = defaultdict(list)
for _,r in df.iterrows():
    s, tt, q, p = r["Symbol"], r["Trade Type"], int(r["Quantity"]), r["Price"]
    abre = "Buy" in tt
    d = decod(s)
    mult = 100 if d else 1
    flujo = (-1 if abre else 1) * q * p * mult
    pos[s].append((r["Date"], abre, q, p, mult))
    if d:
        print(f"   {r['Date']:%d-%b-%y} {d['tk']:>6} {d['tipo']:>5} {d['k']:>8.1f} "
              f"{d['venc']:>12} {'COMPRA' if abre else 'VENDE':>8} {q:>5} {p:>8.2f} "
              f"{flujo:>+10,.0f}$")
    else:
        print(f"   {r['Date']:%d-%b-%y} {s:>6} {'accion':>5} {'':>8} {'':>12} "
              f"{'COMPRA' if abre else 'VENDE':>8} {q:>5} {p:>8.2f} {flujo:>+10,.0f}$")

print()

         fecha subyacente  tipo   strike        vence   accion  cant   precio    efectivo
   22-Sep-26   AAPL   put    345.0  25-sep-2026   COMPRA     1     5.80       -580$
   23-Sep-26   AAPL   put    345.0  25-sep-2026    VENDE     1     7.65       +765$
   30-Sep-26   AAPL accion                         COMPRA   100   330.80    -33,080$
   30-Sep-26   AAPL   put    320.0  30-oct-2026   COMPRA     1     4.90       -490$
   01-Oct-26   NVDA  call    227.5   9-oct-2026   COMPRA     1     5.92       -592$
   06-Oct-26   AAPL  call    332.5  16-oct-2026   COMPRA     1     5.25       -525$
   06-Oct-26    SLV  call     55.0  16-oct-2026   COMPRA     1     1.26       -126$
   06-Oct-26   NVDA  call    227.5   9-oct-2026    VENDE     1    14.98     +1,498$
   13-Oct-26   NVDA   put    225.0  23-oct-2026   COMPRA     2     3.90       -780$
   13-Oct-26    JPM  call    335.0  23-oct-2026   COMPRA     1     6.40       -640$
   13-Oct-26    SPY  call    778.0  23-oct-2026   COMPRA     1     5.

In [7]:
print("="*96)
print("B. RESULTADO POR POSICION CERRADA")

B. RESULTADO POR POSICION CERRADA


In [8]:
print("="*96)
print(f"   {'instrumento':>16} {'descripcion':>34} {'costo':>10} {'ingreso':>10} "
      f"{'P&L':>10} {'retorno':>9}")
tot_c = tot_i = 0.0
res = []
for s, ops in pos.items():
    c = sum(q*p*m for _,a,q,p,m in ops if a)
    i = sum(q*p*m for _,a,q,p,m in ops if not a)
    if c == 0 or i == 0: continue
    d = decod(s)
    desc = (f"{d['tk']} {d['tipo']} {d['k']:g} vence {d['venc']}" if d
            else f"{s} acciones al contado")
    tot_c += c; tot_i += i
    res.append((s, desc, c, i, i-c))
for s,desc,c,i,pl in sorted(res, key=lambda x:-x[4]):
    print(f"   {s:>16} {desc:>34} {c:>9,.0f}$ {i:>9,.0f}$ {pl:>+9,.0f}$ {pl/c*100:>+8.1f}%")
print(f"   {'':>16} {'':>34} {'-'*10} {'-'*10} {'-'*10}")
print(f"   {'TOTAL':>16} {'':>34} {tot_c:>9,.0f}$ {tot_i:>9,.0f}$ {tot_i-tot_c:>+9,.0f}$ "
      f"{(tot_i-tot_c)/tot_c*100:>+8.1f}%")

print()

        instrumento                        descripcion      costo    ingreso        P&L   retorno
               AAPL           AAPL acciones al contado    33,080$    34,615$    +1,535$     +4.6%
     NVDA2609J227.5   NVDA call 227.5 vence 9-oct-2026       592$     1,498$      +906$   +153.0%
     AAPL2616J332.5  AAPL call 332.5 vence 16-oct-2026       525$     1,320$      +795$   +151.4%
        SPY2623V778      SPY put 778 vence 23-oct-2026       459$     1,131$      +672$   +146.4%
       NVDA2623V225     NVDA put 225 vence 23-oct-2026       780$     1,430$      +650$    +83.3%
       AAPL2625U345     AAPL put 345 vence 25-sep-2026       580$       765$      +185$    +31.9%
         SLV2616J55      SLV call 55 vence 16-oct-2026       126$        44$       -82$    -65.1%
        JPM2623J335     JPM call 335 vence 23-oct-2026       640$       395$      -245$    -38.3%
       AAPL2630V320     AAPL put 320 vence 30-oct-2026       490$        82$      -408$    -83.3%
         TLT2623V78 

In [9]:
print("="*96)
print("C. CONTRASTE CON EL SALDO DE LA CUENTA")

C. CONTRASTE CON EL SALDO DE LA CUENTA


In [10]:
print("="*96)
VAL, CAP = 1003085.00, 1000000.00
print(f"   P&L reconstruido de las operaciones   {tot_i-tot_c:>+12,.2f} USD")
print(f"   Variacion observada de la cuenta      {VAL-CAP:>+12,.2f} USD")
print(f"   Diferencia                            {(VAL-CAP)-(tot_i-tot_c):>+12,.2f} USD")
print()
print(f"   Capital desplegado en total           {tot_c:>12,.2f} USD")
print(f"   Retorno sobre capital arriesgado      {(tot_i-tot_c)/tot_c*100:>+11.2f} %")
print(f"   Retorno sobre capital total           {(VAL-CAP)/CAP*100:>+11.2f} %")
print(f"   Operaciones cerradas                  {len(res):>12d}")
gan = sum(1 for *_,pl in res if pl>0)
print(f"   Aciertos                              {gan:>12d}  ({gan/len(res)*100:.0f}%)")
pos_pl = [pl for *_,pl in res if pl>0]; neg_pl = [pl for *_,pl in res if pl<=0]

print(f"   Ganancia media de los aciertos        {sum(pos_pl)/len(pos_pl):>+12,.0f} USD")
print(f"   Perdida media de los errores          {sum(neg_pl)/len(neg_pl):>+12,.0f} USD")
print(f"   Razon ganancia/perdida                "
      f"{abs(sum(pos_pl)/len(pos_pl))/abs(sum(neg_pl)/len(neg_pl)):>12.2f} a 1")

   P&L reconstruido de las operaciones      +3,085.00 USD
   Variacion observada de la cuenta         +3,085.00 USD
   Diferencia                                   +0.00 USD

   Capital desplegado en total              38,320.00 USD
   Retorno sobre capital arriesgado            +8.05 %
   Retorno sobre capital total                 +0.31 %
   Operaciones cerradas                            11
   Aciertos                                         6  (55%)
   Ganancia media de los aciertos                +790 USD
   Perdida media de los errores                  -332 USD
   Razon ganancia/perdida                        2.38 a 1
